# OFFBALL — Exportar K-Means-2 desde tu Colab actual

Inserta la siguiente celda **debajo de la última celda ejecutada** en `OFFBALL_01_Exploracion.ipynb`, donde siguen disponibles `train`, `val`, `X_train`, `X_val`, `scaler` y `variables_base`.

No utiliza test (2025/26), ni valores de mercado. Genera los archivos para Byron a partir de los datos ya cargados en memoria, sin solicitar de nuevo el ZIP original.

In [ ]:
import json
from pathlib import Path
from zipfile import ZipFile

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from google.colab import files

# Exigir que los datos de la sesión sean los mismos del experimento auditado
assert list(variables_base) == ["goles_90", "asistencias_90", "amarillas_90"]
assert set(train["season"].unique()) == {2023}
assert set(val["season"].unique()) == {2024}
assert X_train.shape == (len(train), 3) and X_val.shape == (len(val), 3)
assert len(train) == 1816 and len(val) == 1814, "El dataset cambió; revisar antes de exportar"
assert not train.duplicated(["player_id", "season", "competition_id"]).any()
assert not val.duplicated(["player_id", "season", "competition_id"]).any()

# Reentrenar exactamente la configuración de K-Means-2 del notebook
kmeans_b = KMeans(n_clusters=2, random_state=42, n_init=20).fit(X_train)
train_labels = kmeans_b.predict(X_train)
val_labels = kmeans_b.predict(X_val)

# Inferir significado de cada etiqueta solo a partir de TRAIN
resumen = train[["position"] + variables_base].copy()
resumen["cluster_kmeans_2"] = train_labels
puntuacion = resumen.groupby("cluster_kmeans_2")[["goles_90", "asistencias_90"]].mean().sum(axis=1)
cluster_alto = int(puntuacion.idxmax())
perfiles = {cluster_alto: "Mayor producción ofensiva", 1-cluster_alto: "Menor producción ofensiva"}

carpeta = Path("OFFBALL_KMeans2_Para_Byron")
carpeta.mkdir(exist_ok=True)
claves = ["player_id", "season", "competition_id"]

def guardar_asignaciones(df, X, etiquetas, nombre):
    t = df[claves].copy()
    t["cluster_kmeans_2"] = etiquetas.astype(int)
    t["kmeans_alta_produccion"] = (etiquetas == cluster_alto).astype(int)
    t["perfil_kmeans_2"] = [perfiles[int(x)] for x in etiquetas]
    t["distancia_centroide"] = np.linalg.norm(X - kmeans_b.cluster_centers_[etiquetas], axis=1)
    t.to_csv(carpeta / nombre, index=False)

guardar_asignaciones(train, X_train, train_labels, "kmeans2_train_2023.csv")
guardar_asignaciones(val, X_val, val_labels, "kmeans2_val_2024.csv")

tabla = resumen.groupby("cluster_kmeans_2").agg(
    observaciones=("goles_90", "size"),
    goles_90=("goles_90", "mean"),
    asistencias_90=("asistencias_90", "mean"),
    amarillas_90=("amarillas_90", "mean")
)
pos = pd.crosstab(resumen["cluster_kmeans_2"], resumen["position"], normalize="index")*100
tabla = tabla.join(pos.add_prefix("pct_"))
tabla["perfil"] = tabla.index.map(perfiles)
tabla.to_csv(carpeta / "resumen_kmeans2_train.csv")

joblib.dump({
    "scaler": scaler,
    "modelo": kmeans_b,
    "variables": list(variables_base),
    "claves": claves,
    "cluster_alta_produccion": cluster_alto,
    "perfiles": perfiles,
    "season_train": 2023,
    "season_val": 2024,
    "min_minutos": 450,
}, carpeta / "kmeans2_pipeline.joblib")

metricas = {
    "silueta_train": float(silhouette_score(X_train, train_labels)),
    "silueta_val": float(silhouette_score(X_val, val_labels)),
    "inercia_train": float(kmeans_b.inertia_),
    "n_train": len(train), "n_val": len(val),
    "cluster_alta_produccion": cluster_alto,
    "sklearn_version": sklearn.__version__
}
(carpeta / "metricas_kmeans2.json").write_text(json.dumps(metricas, indent=2), encoding="utf-8")
print(metricas)
print("\nDistribución de clusters:\n", tabla.round(3))

ruta_zip = "OFFBALL_KMeans2_Modelo_y_Asignaciones.zip"
with ZipFile(ruta_zip, "w") as z:
    for f in sorted(carpeta.iterdir()):
        z.write(f, arcname=f.name)
print("\nPaquete listo:", ruta_zip)
files.download(ruta_zip)
